# Comparing results

The code within this workbook reads data from *.sin files, both nodal displacments and stresses



In [ ]:
import dnv.net.runtime
from dnv.sesam.sifapi.core import ISifData
from dnv.sesam.sifapi.io import SesamDataFactory
import os
import numpy as np
import pandas as pd
from qats import TsDB, TimeSeries
import glob


## Define functions for getting data from sin file

Data is returned as numpy arrays

In [ ]:


def get_series_for_element(sin_file,internal_node_nos):
    """
    Reads nodal displacement data (RVNODDIS) of all result cases and given FE node in the given SIN file
    
    Note: switching the order in the double for-loop is significantly slower
    
    """

    root, file = os.path.split(sin_file) 
    run_root = os.getcwd()
    os.chdir(root)
    if not isinstance(internal_node_nos, list):
        internal_node_nos = [internal_node_nos]

    with SesamDataFactory.CreateReader('.', file) as adapter:
        adapter.CreateModel()
        lc_no = adapter.GetCount('RDRESREF')
        data = np.empty((len(internal_node_nos), 6, lc_no))
        
        for j, node_no in enumerate(internal_node_nos):
            for i in range(lc_no):
                res = adapter.ReadExt('RVNODDIS', [i+1, node_no])
               
                res = [e for e in res]
                data[j,:,i] = res[6:]

    os.chdir(run_root)
    return data





In [ ]:


def get_stress_series_for_element(sin_file,element_input):
    """
    Get stress series for a result point in a element.

    sin_file: path to the SIN file
    element_input: list of tuples with element number and result point number, e.g. [(24,1),(24,2)]
    
    Returns: numpy array with shape (len(element_input), 3, lc_no) where lc_no is the number of load cases, 
        containing a length 3 vector for each stress component (sig-xx, sig-yy, tau) for each element and load case.

    Limitations: only work for sesam element type 24 and 25 (thin shell first order)
    """
    
    root, file = os.path.split(sin_file) 
    run_root = os.getcwd()
    os.chdir(root)
    no_stres_comp = 3
    if not isinstance(element_input, list):
        element_input = [element_input]
    with SesamDataFactory.CreateReader('.', file) as adapter:
        adapter.CreateModel()
        lc_no = adapter.GetCount('RDRESREF')
        data = np.empty((len(element_input), no_stres_comp, lc_no))
        
        for j, el_info in enumerate(element_input):
            el_no, resultpoint = el_info
            
            for i in range(lc_no):
                res = adapter.ReadExt('RVSTRESS', [i+1, el_no])
               
                res = [e for e in res]
                start = 6+(resultpoint-1)*no_stres_comp
                stop =  6+(resultpoint)*no_stres_comp
                try:
                    data[j,:,i] = res[start:stop]
                except ValueError:
                    data[j,:,i] = np.zeros((3,))
    os.chdir(run_root)
    return data





## Find files to compare

In [ ]:
resfiles = glob.glob('../Truncated/**/*_R3.sin',recursive=True)

Extract nodal displacements for selected nodes, from all selected sin files

In [ ]:

nodes = [9653, 5017, 6759, 9811, 7973, 1181]
datas = {}
for resfile in resfiles:
    if "Lowres" in resfile or "stress" in resfile:
        continue

    hierachy = resfile.split('\\')
    dlc = hierachy[-2].split('-')[0]
    meth = hierachy[-3]
    data =  get_series_for_element(resfile, nodes)
    if not meth in datas.keys():
        datas[meth] = {}
    datas[meth][dlc] = data

Export the data into QATS files. 
See [QATS github page](https://github.com/dnvgl/qats) for more info

In [ ]:
os.chdir(r'E:\Whitpaper-TD-Methods\Analyses\comparison')
db = TsDB()


for meth in datas.keys():
    for dlc in datas[meth].keys():
        data = datas[meth][dlc]
        db = TsDB()
        t = np.arange(0,data.shape[2])
        for i, node in enumerate(nodes):
            for j, response_type in enumerate(['X','Y','Z', 'Rx','Ry','Rz']):
                ts = TimeSeries(f'{node}-{response_type}',t,data[i,j,:])
                db.add(ts)
        db.export(f'response-{meth}-{dlc}.ts')



## Look at stress

In [ ]:
# elno, resultpoint, 
elements = [(28079,2), (28079,3),(28078,1), ]
stressdatas = {}
for resfile in resfiles:
    if "Lowres" in resfile:
        continue

    hierachy = resfile.split('\\')
    dlc = hierachy[-2].split('-')[0]
    meth = hierachy[-4]
    data =  get_stress_series_for_element(resfile, elements)
    if not meth in stressdatas.keys():
        stressdatas[meth] = {}
    stressdatas[meth][dlc] = data
    

Export to QATS


In [ ]:

os.chdir(r'E:\Whitpaper-TD-Methods\Analyses\comparison')
for meth in stressdatas.keys(): 
    for dlc in stressdatas[meth].keys():
        data = stressdatas[meth][dlc]
        db = TsDB()
        t = np.arange(0,data.shape[2])
        for i, el_info in enumerate(elements):
            el_no, resultpoint = el_info
            for j, response_type in enumerate(['sig_xx','sig_yy','tau_xy']):
                ts = TimeSeries(f'El{el_no}-RP{resultpoint}-{response_type}',t,data[i,j,:])
                db.add(ts)
        db.export(f'stress-{meth[3:]}-{dlc}.ts')

Plot data using pandas


In [ ]:
df = pd.DataFrame(stressdatas[meth][dlc][1,:,:].T, columns=['sig_xx','sig_yy','tau_xy'])
df.plot()